In [ ]:
import wandb
import pandas as pd
import numpy as np
import json
import os
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Set plotting style for better aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['figure.dpi'] = 100

In [ ]:
# --- TODO: SET YOUR WANDB PROJECT PATH HERE ---
# This is usually in the format "username/project-name"
# You can find it in the URL of your W&B project page.
WANDB_PROJECT_PATH = "sandesh1122bhattarai-the-university-of-southern-mississippi/fl-baseline-research"

# --- TODO: SET THE LOCAL DIRECTORY TO STORE ARTIFACTS ---
ARTIFACTS_DIR = "./wandb_artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print(f"Configuration set:")
print(f" - W&B Project: {WANDB_PROJECT_PATH}")
print(f" - Artifacts Directory: {ARTIFACTS_DIR}")

In [ ]:
print("Fetching run data from W&B... This may take a moment.")

# Initialize the W&B API
api = wandb.Api()
runs = api.runs(WANDB_PROJECT_PATH)

all_run_data = []

for run in runs:
    # --- Get config and summary ---
    run_config = {k: v for k, v in run.config.items() if not k.startswith('_')}
    
    # --- Download the summary artifact ---
    summary_artifact_name = f"{run.name}_summary:latest"
    try:
        artifact = api.artifact(f"{WANDB_PROJECT_PATH}/{summary_artifact_name}", type="run-summary")
        summary_path = artifact.download(root=ARTIFACTS_DIR)
        with open(os.path.join(summary_path, "summary.json")) as f:
            summary_metrics = json.load(f)
    except wandb.errors.CommError:
        print(f"⚠️ Could not find summary artifact for run: {run.name}. Using run.summary instead (may be incomplete).")
        summary_metrics = {k: v for k, v in run.summary._json_dict.items()}

    # Combine config and summary into a single record
    record = {**run_config, **summary_metrics}
    all_run_data.append(record)

# Convert to a Pandas DataFrame for easy analysis
df = pd.DataFrame(all_run_data)

# --- Data Cleaning and Feature Engineering ---
# Create a malicious ratio column for break-point analysis
df['malicious_ratio'] = df['num_malicious'] / df['num_partitions']

# Ensure numeric types
for col in ['best_accuracy', 'final_accuracy', 'rounds_to_convergence', 'avg_aggregation_time']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"\n✅ Successfully fetched and processed data for {len(df)} runs.")
print("\nDataFrame Preview:")
df.head()

In [ ]:
# --- Filter for the baseline attack scenario (n=10, f=3) ---
baseline_df = df[df['run_name'].str.contains('attack') & (df['num_partitions'] == 10) & (df['num_malicious'] == 3)].copy()

if not baseline_df.empty:
    # --- Calculate the Computational Cost Ratio ---
    fedavg_agg_time = baseline_df[baseline_df['strategy'] == 'FedAvg']['avg_aggregation_time'].iloc[0]
    baseline_df['comp_cost_ratio'] = baseline_df['avg_aggregation_time'] / fedavg_agg_time
    
    plt.figure(figsize=(12, 7))
    ax = sns.barplot(data=baseline_df, x='strategy', y='comp_cost_ratio', palette='viridis')
    
    ax.set_title('Computational Cost of Robustness vs. FedAvg (Attack Scenario)', fontsize=16, fontweight='bold')
    ax.set_xlabel('Aggregation Strategy', fontsize=12)
    ax.set_ylabel('Cost Ratio (Higher is Slower)', fontsize=12)
    ax.axhline(1.0, color='r', linestyle='--', label='FedAvg Baseline')
    ax.legend()
    
    # Add labels to bars
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1f}x", 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='center', 
                    xytext=(0, 9), 
                    textcoords='offset points')
    
    plt.xticks(rotation=15)
    plt.show()
else:
    print("Could not find baseline attack runs to generate 'Cost of Robustness' plot.")

In [ ]:
# --- Use the same baseline_df from the previous cell ---
if not baseline_df.empty:
    plt.figure(figsize=(12, 8))
    
    # Use total runtime as the cost metric
    plot_df = baseline_df.dropna(subset=['avg_aggregation_time', 'best_accuracy'])
    
    ax = sns.scatterplot(data=plot_df, x='avg_aggregation_time', y='best_accuracy', hue='strategy', s=200, style='strategy', palette='deep', alpha=0.9)
    
    # Annotate points with strategy names
    for i, row in plot_df.iterrows():
        plt.text(row['avg_aggregation_time'] * 1.05, row['best_accuracy'], row['strategy'], fontsize=9, va='center')
        
    ax.set_title('Accuracy vs. Aggregation Time Trade-off (Attack Scenario)', fontsize=16, fontweight='bold')
    ax.set_xlabel('Average Aggregation Time per Round (seconds)', fontsize=12)
    ax.set_ylabel('Best Achieved Accuracy', fontsize=12)
    ax.legend(title='Strategy', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Add helper text
    plt.text(0.95, 0.05, '← Cheaper | Slower →', ha='right', va='bottom', transform=ax.transAxes, fontsize=10, color='gray')
    plt.text(0.05, 0.95, '↑ More Accurate\n↓ Less Accurate', ha='left', va='top', transform=ax.transAxes, fontsize=10, color='gray')
    
    plt.tight_layout()
    plt.show()
else:
    print("Could not find baseline attack runs to generate 'Accuracy vs. Cost' plot.")

In [ ]:
# --- Filter for scalability runs ---
scalability_df = df[df['run_name'].str.contains('scalability')].sort_values('num_partitions')

if not scalability_df.empty:
    plt.figure(figsize=(10, 6))
    ax = sns.lineplot(data=scalability_df, x='num_partitions', y='avg_aggregation_time', hue='strategy', marker='o', style='strategy', markersize=8)
    
    ax.set_title('Scalability Analysis: Aggregation Time vs. Number of Clients', fontsize=16, fontweight='bold')
    ax.set_xlabel('Number of Clients (n)', fontsize=12)
    ax.set_ylabel('Average Aggregation Time (seconds, log scale)', fontsize=12)
    ax.set_yscale('log') # Use log scale to show the O(n^2) difference clearly
    ax.get_yaxis().set_major_formatter(mtick.ScalarFormatter()) # Format y-axis nicely
    
    plt.legend(title='Strategy')
    plt.show()
else:
    print("Could not find scalability runs to generate plot.")

In [ ]:
# --- Filter for breakpoint runs ---
breakpoint_df = df[df['run_name'].str.contains('breakpoint')].sort_values('malicious_ratio')

if not breakpoint_df.empty:
    plt.figure(figsize=(12, 7))
    ax = sns.lineplot(data=breakpoint_df, x='malicious_ratio', y='best_accuracy', hue='strategy', marker='o', style='strategy', markersize=8)
    
    ax.set_title('Break-Point Analysis: Accuracy vs. Malicious Client Ratio', fontsize=16, fontweight='bold')
    ax.set_xlabel('Ratio of Malicious Clients (f/n)', fontsize=12)
    ax.set_ylabel('Best Achieved Accuracy', fontsize=12)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.axhline(0.1, color='k', linestyle=':', label='Random Guess (10%)')
    
    plt.legend(title='Strategy')
    plt.show()
else:
    print("Could not find breakpoint runs to generate plot.")